# Credit Risk — PD, LGD, EL and IRB Capital

Basel III / CRR — Foundation IRB approach: probability of default,
loss given default, expected loss provisioning, and unexpected-loss
regulatory capital.

| Component | Formula | Regulatory ref |
|---|---|---|
| Expected Loss | EL = PD × LGD × EAD | CRR Art. 5 |
| Unexpected Loss | K = IRB formula | CRR Art. 153 |
| Risk-Weighted Assets | RWA = K × 12.5 × EAD × MA | CRR Art. 92 |

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from banking_risk.credit_risk import (
    # PD
    RATING_PD_TABLE,
    PD_Estimate,
    Rating_PD_Model,
    Logistic_PD_Model,
    # LGD
    Collateral_Type,
    LGD_FLOORS,
    LGD_Estimate,
    CRR_LGD_Model,
    # EL / IRB
    EL_Position,
    EL_Result,
    Expected_Loss_Calculator,
)
from banking_risk.utils.reporting import Dark_Style

## 2. Probability of Default

### 2a. Rating-based PD (through-the-cycle)

Maps S&P-style ratings to long-run 1-year PDs calibrated to Basel/EBA
anchor values. The CRR Art. 163 minimum floor of **0.03 %** applies to
all non-defaulted exposures.

In [ ]:
# Full rating PD table
pd_table = pd.Series(RATING_PD_TABLE, name="1Y PD").rename_axis("rating")
pd_table_pct = (pd_table * 100).round(4).to_frame("1Y PD (%)")
pd_table_pct

In [ ]:
rating_model = Rating_PD_Model()
sample_ratings = ["A", "BBB", "BBB-", "BB", "B", "CCC", "D"]

rows = []
for r in sample_ratings:
    est = rating_model.predict(r)
    rows.append({"rating": r, "PD (%)": round(est.pd * 100, 4), "model": est.model})

pd.DataFrame(rows).set_index("rating")

### 2b. Logistic regression PD (point-in-time)

Point-in-time PD from financial statement features. Typical features
in corporate credit scoring and the expected sign of each coefficient:

| Feature | Sign | Interpretation |
|---|---|---|
| leverage (D/EBITDA) | + | higher leverage → higher risk |
| interest_coverage (EBIT/Interest) | − | better coverage → lower risk |
| roa (NI/Assets) | − | more profitable → lower risk |
| size (log assets €M) | − | larger firms → lower risk (diversification) |

In [ ]:
logistic_model = Logistic_PD_Model(
    coefficients={
        "leverage"          :  0.35,
        "interest_coverage" : -0.20,
        "roa"               : -2.50,
        "size"              : -0.15,
    },
    intercept=-4.5,
)

# Three stylised borrowers
borrowers = {
    "Investment_Grade" : {"leverage": 2.5,  "interest_coverage": 8.0, "roa": 0.08, "size": 4.5},
    "Sub_Investment"   : {"leverage": 5.0,  "interest_coverage": 3.5, "roa": 0.04, "size": 3.8},
    "Stressed"         : {"leverage": 9.0,  "interest_coverage": 1.5, "roa": -0.01, "size": 3.0},
}

rows = []
for name, features in borrowers.items():
    est = logistic_model.predict(features)
    rows.append({
        "name"      : name,
        "leverage"  : features["leverage"],
        "int_cover" : features["interest_coverage"],
        "roa"       : features["roa"],
        "log_odds"  : round(est.log_odds, 3),
        "PD (%)"    : round(est.pd * 100, 4),
    })

pd.DataFrame(rows).set_index("name")

In [ ]:
# Sensitivity: PD vs leverage for the Sub_Investment borrower
Dark_Style().apply()
p = Dark_Style().palette

leverages = np.linspace(1, 15, 100)
base      = {k: v for k, v in borrowers["Sub_Investment"].items()}
pds       = [logistic_model.predict({**base, "leverage": lev}).pd * 100 for lev in leverages]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(leverages, pds, color=p["cyan"], lw=2)
for name, features in borrowers.items():
    est = logistic_model.predict(features)
    ax.scatter(features["leverage"], est.pd * 100, s=80, zorder=5,
               color=p["amber"], label=f"{name} ({est.pd*100:.2f}%)")
ax.set_xlabel("Leverage (D/EBITDA)")
ax.set_ylabel("1Y PD (%)")
ax.set_title("Logistic PD vs leverage — holding other features constant",
             color=p["text_title"])
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Loss Given Default

CRR Art. 228–230 collateral haircut approach. The regulatory LGD floors
prevent setting LGD below the prescribed minimums regardless of collateral.

| Collateral type | LGD floor | LGD if uncollateralised |
|---|---|---|
| Residential real estate | 10 % | 45 % |
| Commercial real estate | 15 % | 45 % |
| Financial collateral | 0 % | 45 % |
| Unsecured senior | 45 % | 45 % |
| Subordinated | 75 % | 75 % |

In [ ]:
lgd_model = CRR_LGD_Model()
ead       = 1_000_000

rows = []
for ct in Collateral_Type:
    for coverage_pct in [0, 25, 50, 75, 100]:
        collateral = ead * coverage_pct / 100
        est = lgd_model.estimate(ead, collateral, ct, haircut=0.0)
        rows.append({
            "collateral_type": ct.value,
            "coverage_%"     : coverage_pct,
            "coverage_ratio" : round(est.coverage_ratio, 4),
            "LGD (%)"        : round(est.lgd * 100, 2),
            "floor (%)"      : round(est.lgd_floor * 100, 2),
        })

lgd_df = pd.DataFrame(rows)
lgd_df.pivot(index="collateral_type", columns="coverage_%", values="LGD (%)").round(2)

In [ ]:
# Haircut effect: residential RE, 80 % collateral, varying haircuts
haircuts = np.linspace(0, 0.6, 100)
lgds_no_hc = [lgd_model.estimate(ead, 800_000, Collateral_Type.RESIDENTIAL_RE, h).lgd for h in haircuts]
lgds_hi_ead = [lgd_model.estimate(ead, 600_000, Collateral_Type.COMMERCIAL_RE,   h).lgd for h in haircuts]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(haircuts * 100, [x * 100 for x in lgds_no_hc],  color=p["cyan"],  lw=2, label="Residential RE, 80% collateral")
ax.plot(haircuts * 100, [x * 100 for x in lgds_hi_ead], color=p["amber"], lw=2, label="Commercial RE, 60% collateral")
ax.axhline(LGD_FLOORS[Collateral_Type.RESIDENTIAL_RE] * 100, color=p["cyan"],  lw=1, ls=":", alpha=0.7)
ax.axhline(LGD_FLOORS[Collateral_Type.COMMERCIAL_RE]  * 100, color=p["amber"], lw=1, ls=":", alpha=0.7)
ax.set_xlabel("Haircut (%)")
ax.set_ylabel("LGD (%)")
ax.set_title("LGD vs collateral haircut — floor enforced (dotted)", color=p["text_title"])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Expected Loss and IRB Capital

EL = PD × LGD × EAD — the statistical mean loss, provisioned under IFRS 9.

The IRB unexpected-loss capital K covers the loss at the 99.9 % confidence
level above EL. K is driven by the **asset correlation R**, which captures
systemic risk: at low PD (investment grade), R is high (~24 %) because
corporate defaults are more driven by macro cycles than idiosyncratic events.

In [ ]:
# Stylised corporate loan portfolio
portfolio = [
    EL_Position("Corp_A_A",      ead=10_000_000, pd=0.0009, lgd=0.10, maturity_years=3.0),
    EL_Position("Corp_B_BBB",    ead= 8_000_000, pd=0.0025, lgd=0.45, maturity_years=5.0),
    EL_Position("Corp_C_BBB",    ead= 6_000_000, pd=0.0050, lgd=0.40, maturity_years=4.0),
    EL_Position("Corp_D_BB",     ead= 4_000_000, pd=0.0100, lgd=0.45, maturity_years=3.0),
    EL_Position("Corp_E_B",      ead= 2_000_000, pd=0.0500, lgd=0.60, maturity_years=2.0),
    EL_Position("Corp_F_CCC",    ead=   500_000, pd=0.2000, lgd=0.75, maturity_years=1.5),
]

el_calc = Expected_Loss_Calculator()
result  = el_calc.compute(portfolio)

display = result.detail.copy()
display["el_K"]  = display["el"]  / 1e3
display["rwa_M"] = display["rwa"] / 1e6
display["K_pct"] = display["K"]   * 100
display["pd_pct"] = display["pd"] * 100
display[["ead", "pd_pct", "lgd", "maturity_years", "el_K", "K_pct", "rwa_M"]].rename(
    columns={"pd_pct": "PD %", "K_pct": "K %", "el_K": "EL (€K)", "rwa_M": "RWA (€M)"}
).round(4)

In [ ]:
print(f"Total EAD   : EUR {result.total_ead:>12,.0f}")
print(f"Total EL    : EUR {result.total_el:>12,.0f}  ({result.el_ratio:.4%} of EAD)")
print(f"Total RWA   : EUR {result.total_rwa:>12,.0f}  ({result.total_rwa/result.total_ead:.1%} of EAD)")
print(f"Capital req : EUR {result.total_rwa * 0.08:>12,.0f}  (8% × RWA — CRR Art. 92)")

In [ ]:
# EL vs IRB capital K comparison
Dark_Style().apply()
p = Dark_Style().palette

detail    = result.detail
names     = detail.index.tolist()
el_pct    = detail["el"]  / detail["ead"] * 100
k_pct     = detail["K"]   * 100
x         = np.arange(len(names))
w         = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, el_pct, w, color=p["cyan"],  alpha=0.85, label="EL (% of EAD)")
ax.bar(x + w/2, k_pct,  w, color=p["amber"], alpha=0.85, label="IRB capital K (% of EAD)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("% of EAD")
ax.set_title("Expected Loss vs IRB Unexpected-Loss Capital K — CRR Art. 153",
             color=p["text_title"], fontweight="bold")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. IRB asset correlation and the capital formula

The asset correlation R(PD) is the key driver of the capital formula.
At low PD, R ≈ 24 % (returns are highly correlated — investment grade
defaults are rare but systemic when they occur). At high PD, R → 12 %
(idiosyncratic risk dominates for distressed names).

> R = 0.12 × h + 0.24 × (1 − h)  where h = (1 − e^(−50 PD)) / (1 − e^(−50))

In [ ]:
from banking_risk.credit_risk.el import _asset_correlation, _irb_capital

pds = np.logspace(-4, -0.3, 200)   # 0.01% to 50%

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: asset correlation R(PD)
rs = [_asset_correlation(pd) * 100 for pd in pds]
ax1.semilogx(pds * 100, rs, color=p["cyan"], lw=2)
ax1.axhline(12, color=p["text_muted"], lw=0.8, ls="--", label="R_min = 12%")
ax1.axhline(24, color=p["text_muted"], lw=0.8, ls=":", label="R_max = 24%")
ax1.set_xlabel("PD (%)")
ax1.set_ylabel("Asset correlation R (%)")
ax1.set_title("CRR Art. 153 — Asset correlation R(PD)", color=p["text_title"])
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Panel 2: K vs PD for two LGD levels
for lgd, color, label in [(0.45, p["amber"], "LGD = 45%"), (0.25, p["green"], "LGD = 25%")]:
    ks = [_irb_capital(pd, lgd) * 100 for pd in pds]
    ax2.semilogx(pds * 100, ks, color=color, lw=2, label=label)
ax2.set_xlabel("PD (%)")
ax2.set_ylabel("K (% of EAD)")
ax2.set_title("IRB capital K vs PD — CRR Art. 153 (M = 2.5Y, MA omitted)",
              color=p["text_title"])
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig.suptitle("Foundation IRB — Asset Correlation and Capital Formula",
             color=p["text_title"], fontweight="bold")
fig.tight_layout()
plt.show()